# 02 · Train the classifier on Kaggle

All training code lives in the `neurolens_ml` package; this notebook only runs it on a Kaggle GPU.

1. Clone the repo and install the package.
2. Run `python -m neurolens_ml.train` with a YAML config from `ml/configs/`.
3. Show the curves, val confusion matrix and metrics.

**Kaggle setup:** GPU T4 x2, Internet on, inputs `masoudnickparvar/brain-tumor-mri-dataset` (+ `splits.csv` dataset, optional: the repo has one). Run with *Save Version → Save & Run All*. Outputs land in `/kaggle/working`; download them with *Output → Download all*.

The test split is never opened here (tune on val only).

In [ ]:
import importlib.util
import json
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/sanjardevx/neurolens.git"
BRANCH = "develop"
CONFIG = "ml/configs/cls_b3.yaml"  # cls_b0.yaml for the small baseline
REPO_DIR = Path(os.environ.get("NEUROLENS_REPO_DIR", "/tmp/neurolens"))  # outside /kaggle/working, so it's not in the outputs
OUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("outputs")
EXTRA_ARGS = os.environ.get("NEUROLENS_ARGS", "").split()  # only for local smoke tests (fake data paths, --smoke)

In [ ]:
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
commit = subprocess.run(["git", "-C", str(REPO_DIR), "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
if importlib.util.find_spec("neurolens_ml") is None:
    # --no-deps: Kaggle already has torch, timm, albumentations, opencv; don't reinstall them
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", str(REPO_DIR / "ml")], check=True)
print("repo commit:", commit)

In [ ]:
cmd = [sys.executable, "-m", "neurolens_ml.train", "--config", str(REPO_DIR / CONFIG), "--out", str(OUT_DIR), *EXTRA_ARGS]
print(" ".join(cmd))
# Stream the training log into the notebook output, line by line
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end="")
assert proc.wait() == 0, "training failed, see the log above"

In [ ]:
from IPython.display import Image, display

import yaml

name = yaml.safe_load((REPO_DIR / CONFIG).read_text())["name"]
metrics = json.loads((OUT_DIR / f"metrics_{name}.json").read_text())
metrics["repo_commit"] = commit  # which code made this model
(OUT_DIR / f"metrics_{name}.json").write_text(json.dumps(metrics, indent=2))
print("best epoch:", metrics["best_epoch"], "| val:", metrics["val"])
for f in [f"curves_{name}.png", f"confusion_val_{name}.png", f"errors_val_{name}.png"]:
    if (OUT_DIR / f).exists():
        display(Image(filename=str(OUT_DIR / f)))